# Phase 5 — Machine Learning Meta-Labeling & Probability Calibration

This interactive research notebook executes the complete **Phase 5 Tabular Meta-Labeling Workflow**:
1. **Primary Model Candidate Generation** from `BreakoutSanityStrategy` or `LiquiditySweepFVGStrategy`.
2. **Marcos López de Prado Triple-Barrier Method** label generation ($y=1$ if TP reached before SL/time).
3. **Purged & Embargoed Chronological Cross-Validation** dataset construction.
4. **Calibrated Tabular Model Training** (Logistic Regression, LightGBM, Gradient Boosting).
5. **Secondary ML Meta-Filter Evaluation** & Brier Score Calibration.

In [ ]:
# =============================================================================
# 0. GOOGLE COLAB / LOCAL REPOSITORY SYNC & SETUP
# =============================================================================
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = "https://github.com/umutergul74/daytrader.git"
REPO_DIR = Path("/content/daytrader")

if IN_COLAB:
    print("🚀 [Google Colab Detected] Initializing Daytrader Platform...")
    if not REPO_DIR.exists():
        print(f"Cloning latest repository from {REPO_URL}...")
        !git clone {REPO_URL} /content/daytrader
    else:
        print("Pulling latest updates from GitHub...")
        !cd /content/daytrader && git pull

    os.chdir(str(REPO_DIR))
    print("Installing dependencies...")
    !pip install -q polars pandas numpy scipy scikit-learn lightgbm xgboost catboost pydantic pydantic-settings typer rich matplotlib pyarrow requests websockets pytest optuna

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    print(f"✓ Environment ready! Working directory: {Path.cwd()}")
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(str(project_root))
    if str(project_root / "src") not in sys.path:
        sys.path.insert(0, str(project_root / "src"))
    print(f"✓ [Local Mode] Working directory: {Path.cwd()}")

import polars as pl
import numpy as np
from quant_platform.data.storage.canonical import CanonicalStorage
from quant_platform.data.timeframes.resampler import CausalResampler
from quant_platform.features.indicators.momentum import compute_rsi
from quant_platform.features.indicators.volatility import compute_atr
from quant_platform.features.microstructure.cvd import CvdEngine
from quant_platform.strategies.baselines.breakout import BreakoutSanityStrategy
from quant_platform.ml.meta_labeling import MetaLabelingEngine
from quant_platform.ml.meta_classifier import MetaClassifierTrainer
from quant_platform.strategies.advanced.meta_strategy import MetaLabeledStrategy

print("✓ ML Meta-Labeling research environment initialized.")

## 1. Prepare Features & Generate Primary Strategy Candidates

In [ ]:
storage = CanonicalStorage()
df_1m = storage.read_symbol("ETHUSDT", start_year=2024, start_month=1).head(3000)

# 1. Resample to 15m
df_15m = CausalResampler.resample(df_1m, target_timeframe="15m")
df_15m = compute_atr(df_15m, period=14)
df_15m = compute_rsi(df_15m, period=14)
df_15m = CvdEngine.compute_kline_delta_features(df_15m)

# 2. Primary Strategy Candidates
primary_strat = BreakoutSanityStrategy(lookback_period=20, risk_reward_ratio=2.0)
candidates = primary_strat.generate_signals(df_15m)

print(f"Generated {len(candidates)} candidate signals from Primary Strategy across {len(df_15m)} 15m bars.")

## 2. Triple-Barrier Labeling & Purged Walk-Forward Split

In [ ]:
dataset = MetaLabelingEngine.compute_triple_barrier_labels(df_15m, candidates, max_holding_bars=20)
X_train, y_train, X_test, y_test = MetaLabelingEngine.purged_walk_forward_split(dataset, train_ratio=0.70, embargo_samples=2)

print(f"Train samples: {len(X_train)} (Positive: {np.mean(y_train):.1%})")
print(f"Test samples (OOS): {len(X_test)} (Positive: {np.mean(y_test):.1%})")

## 3. Train & Calibrate Secondary ML Meta-Model

In [ ]:
trainer = MetaClassifierTrainer(model_type="gradient_boosting")
metrics = trainer.train_and_calibrate(X_train, y_train, X_test, y_test, feature_names=dataset.feature_names)

print(f"OOS ROC-AUC: {metrics.roc_auc:.3f}")
print(f"Brier Score: {metrics.brier_score:.4f}")
print(f"Precision:   {metrics.precision:.1%}")
print(f"Recall:      {metrics.recall:.1%}")

print("\nFeature Importances:")
for fname, imp in sorted(metrics.feature_importances.items(), key=lambda x: x[1], reverse=True):
    print(f" - {fname:25s}: {imp:.4f}")

## 4. Evaluate MetaLabeledStrategy Secondary Filter

In [ ]:
meta_strat = MetaLabeledStrategy(primary_strategy=primary_strat, meta_trainer=trainer, probability_threshold=0.52)
filtered_candidates = meta_strat.generate_signals(df_15m)

print(f"Primary Candidates:  {len(candidates)}")
print(f"ML Filtered Passed:  {len(filtered_candidates)} (-{(len(candidates)-len(filtered_candidates))/max(1,len(candidates)):.1%} filtered)")